# Native-anchor CI — putting a confidence interval under `native ≈ 0.58`

The poster references a single constant: **native Yorùbá speech sits at `tone_i2` ≈ 0.58**. Until now that was a
hand-set anchor with **no provenance, sample size, or CI** — the most exposed number on the poster (it is
load-bearing for "near-native zero-shot" and "values above it are saturation").

This notebook **measures it**: it scores the reference-free tone metric `tone_i2` on a large, multi-speaker set of
**real native Yorùbá recordings** (BibleTTS + OpenSLR-86) and reports `native = mean ± 95% CI`.

**Why this is comparable to the model numbers (0.567 / 0.598 / 0.633).** It uses the *identical* call the
model evals use — `f0_abs_score(..., asr=MMS-yor, mid_ref=None, theta_h/theta_l from f0_abs_calibration.v1.json)`
— and the *identical* aggregation: `tone_i2` = **balanced per-class (H/M/L) recall over all TBU pairs pooled across
clips** (nb07 `_bal_tone_acc`). The native audio is fixed and `f0_abs_score` is deterministic (no generation seed),
so the only uncertainty is *which native clips/speakers* we sampled → a **clip-level bootstrap** is the right CI.

**No OmniVoice, no generation, no A100.** Only the metric + the MMS-yor aligner + native wavs. **GPU: T4 / L4**
(or even CPU, slowly). Runtime ≈ a few minutes for ~250 clips.

**Output:** `native = X ± Y` printed poster-ready, plus `s3://codec-audio-data/tts_data/yoruba/tone_v2/native_anchor.v1.json`.

## 1. Install + restart

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" tqdm matplotlib
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import parselmouth; print("numpy",np.__version__,"| parselmouth",getattr(parselmouth,"VERSION","?"))

## 2. Install `tone-metric` (public package)

In [ ]:
# tone-metric package (public) — same pip-install used by every other notebook
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_f0_abs as f0a            # I2 tone meter -> f0_abs_score (== tone_i2)
from tone_metric.grpo_reward import RewardModels      # CER + MMS-yor aligner used by the metric
try:
    from tone_metric import tone_layer0 as tl         # optional cross-check (balanced_accuracy/aggregate)
except Exception as _e:
    tl = None; print("tone_layer0 cross-check unavailable:", _e)
CODE_DIR = os.path.dirname(tone_metric.__file__)
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)

## 3. Secrets + S3 + scoring stack (MMS-yor aligner + CER) + F0 calibration

In [ ]:
import os, getpass, boto3, torch, numpy as np, json
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass

BUCKET="codec-audio-data"; s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; SR=24000

BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"            # native human recordings (BibleTTS + SLR86)
F0CAL_KEY     ="tts_data/yoruba/tone_v2/f0_abs_calibration.v1.json"  # frozen theta_h/theta_l (same as model evals)
ANCHOR_KEY    ="tts_data/yoruba/tone_v2/native_anchor.v1.json"       # <- we write this
WORK="/content/native_anchor" if os.path.isdir("/content") else "/tmp/native_anchor"; os.makedirs(WORK,exist_ok=True)

rm=RewardModels(device=DEVICE)                                      # loads MMS-yor (aligner+CER); ECAPA unused here
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
print(f"ready | DEVICE {DEVICE} | calibration theta_h={I2_TH} theta_l={I2_TL} mode={I2_MODE} late_frac={I2_LATE}")

## 4. Parameters

`N_NATIVE` clips, capped at `PER_SPK_CAP` per speaker so no single voice dominates the pooled estimate.
Set `SMOKE=True` for a fast ~24-clip dry run first.

In [ ]:
SMOKE       = False     # True -> tiny dry run to confirm it works end-to-end
N_NATIVE    = 24 if SMOKE else 300     # target number of native clips to score
PER_SPK_CAP = 6  if SMOKE else 25      # max clips per speaker_id (spread across voices)
DUR_MIN, DUR_MAX = 3.0, 9.0            # seconds (same band the model eval uses)
N_BOOT      = 2000                     # clip-level bootstrap resamples
ALPHA       = 0.05                     # 95% CI
BOOT_SEED   = 4242
print(f"N_NATIVE={N_NATIVE} PER_SPK_CAP={PER_SPK_CAP} dur[{DUR_MIN},{DUR_MAX}]s n_boot={N_BOOT} CI={int((1-ALPHA)*100)}%")

## 5. Load native clips (spread across speakers & sources)

In [ ]:
import io, json, soundfile as sf, numpy as np
from collections import Counter
clips=[]; per_spk=Counter()
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    d=float(r.get("duration_sec",0.0))
    if not (DUR_MIN<=d<=DUR_MAX): continue
    txt=(r.get("text") or "").strip()
    if not txt: continue
    spk=str(r.get("speaker_id","?")); src=r.get("source","?")
    if per_spk[spk]>=PER_SPK_CAP: continue
    key=r.get("audio_s3_key")
    if not key: continue
    lp=f"{WORK}/clip_{r['id']}.wav"
    try: s3.download_file(BUCKET,key,lp)
    except Exception as e: print("  skip (download)",r['id'],"->",e); continue
    per_spk[spk]+=1
    clips.append(dict(id=r["id"],text=txt,source=src,speaker=spk,path=lp,dur=d))
    if len(clips)>=N_NATIVE: break
assert len(clips)>=8, f"too few native clips ({len(clips)})"
src_counts=Counter(c["source"] for c in clips)
print(f"native clips: {len(clips)} | sources: {dict(src_counts)} | distinct speakers: {len(per_spk)}")
print("per-speaker (top 8):", per_spk.most_common(8))

## 6. Score each native clip with `tone_i2`

Byte-identical to the model-eval call: MMS-yor aligner (`asr=rm.asr ...`), `mid_ref=None`, calibrated
`theta_h/theta_l`. `_bal_tone_acc` is copied verbatim from nb07 so the native number is on the same scale as
0.567 / 0.598 / 0.633.

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm
import numpy as np, soundfile as sf, soxr

def _bal_tone_acc(pairs):                 # VERBATIM from nb07 one_pass -> the model's tone_i2 aggregation
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    recs=[cor[c]/tot[c] for c in tot if tot[c]>0]
    return float(np.mean(recs)) if recs else float("nan")

def _read(path):
    w,sr=sf.read(path,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    if sr!=SR: w=soxr.resample(w,sr,SR).astype("float32"); sr=SR
    return w

def score_native(wav,text):
    logits,n16=rm.asr_logits(wav,SR)                                  # one shared MMS forward
    cer=RewardModels.cer(text,rm.decode_logits(logits))
    i2=f0a.f0_abs_score(wav,SR,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return dict(cer=cer,tone_i2_clip=_bal_tone_acc(pairs),coverage=i2["coverage"],
                n_tbu=i2["n_tbu"],n_scored=i2["n_scored"],pairs=pairs)

records=[]
for c in tqdm(clips,desc="scoring native clips"):
    try:
        w=_read(c["path"]); rec=score_native(w,c["text"])
    except Exception as e:
        print("  score fail",c["id"],"->",e); continue
    records.append({**{k:c[k] for k in ("id","source","speaker","text","dur")},**rec})
n_scored_clips=sum(1 for r in records if r["pairs"])
print(f"\nscored {len(records)} clips ({n_scored_clips} with >=1 tone pair) | "
      f"total TBUs scored: {sum(r['n_scored'] for r in records)}")

## 7. Native anchor + clip-level bootstrap 95% CI

In [ ]:
import random as pyrandom, numpy as np
clip_pairs=[r["pairs"] for r in records if r["pairs"]]                # one list of (pred,target) per clip
all_pairs=[p for cp in clip_pairs for p in cp]
point=_bal_tone_acc(all_pairs)                                        # native tone_i2 (pooled) — matches the model

def boot_ci(clip_pairs, stat, n_boot, alpha, seed):
    rng=pyrandom.Random(seed); N=len(clip_pairs); vals=[]
    for _ in range(n_boot):
        pooled=[p for _ in range(N) for p in clip_pairs[rng.randrange(N)]]  # resample whole CLIPS
        v=stat(pooled)
        if v==v: vals.append(v)
    vals.sort()
    lo=vals[int((alpha/2)*len(vals))]; hi=vals[min(len(vals)-1,int((1-alpha/2)*len(vals)))]
    return lo,hi
lo,hi=boot_ci(clip_pairs,_bal_tone_acc,N_BOOT,ALPHA,BOOT_SEED)
half=(hi-lo)/2.0

clip_scores=[r["tone_i2_clip"] for r in records if r["tone_i2_clip"]==r["tone_i2_clip"]]
clip_mean=float(np.mean(clip_scores)); clip_std=float(np.std(clip_scores,ddof=1)) if len(clip_scores)>1 else 0.0
cov_mean=float(np.mean([r["coverage"] for r in records]))
cer_mean=float(np.mean([r["cer"] for r in records if r["cer"]==r["cer"]]))

# pooled per-class (H/M/L) recall — what drives the balanced score
tot,cor=defaultdict(int),defaultdict(int)
for pp,tt in all_pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
per_class={c:(cor[c]/tot[c] if tot[c] else float("nan")) for c in ("H","M","L")}
per_class_support={c:tot[c] for c in ("H","M","L")}

xcheck=None
if tl is not None:
    try:
        score_dicts=[dict(pred=[p for p,_ in r["pairs"]],target=[t for _,t in r["pairs"]]) for r in records if r["pairs"]]
        xcheck=float(tl.balanced_accuracy(tl.aggregate(score_dicts)))
    except Exception as e:
        print("tone_layer0 cross-check skipped:",e)

print("="*64)
print(f"  NATIVE tone_i2 (pooled, balanced H/M/L) = {point:.3f}")
print(f"  95% CI (clip bootstrap, {N_BOOT}x)       = [{lo:.3f}, {hi:.3f}]   (±{half:.3f})")
print(f"  per-clip mean ± std                      = {clip_mean:.3f} ± {clip_std:.3f}  (n={len(clip_scores)})")
print(f"  per-class recall H/M/L                   = "
      f"{per_class['H']:.2f}/{per_class['M']:.2f}/{per_class['L']:.2f}  support {per_class_support}")
print(f"  mean coverage                            = {cov_mean:.2f}   native CER (MMS-yor) = {cer_mean:.3f}")
if xcheck is not None: print(f"  cross-check tone_layer0.balanced_accuracy= {xcheck:.3f}  (should match pooled)")
print(f"  vs old hand-set anchor 0.58  ->  measured {point:.3f}  (chance 0.333)")
print("="*64)
print(f"\nPOSTER-READY:  native = {point:.2f} \\pm {half:.2f}  "
      f"({len(records)} clips, {len({r['speaker'] for r in records})} speakers, 95% CI)")

## 8. Per-source / per-speaker breakdown (shows it is not one voice)

In [ ]:
from collections import defaultdict
def pooled_by(key):
    grp=defaultdict(list)
    for r in records:
        grp[r[key]].extend(r["pairs"])
    return {k:(_bal_tone_acc(v),len(v)) for k,v in grp.items()}
by_src=pooled_by("source")
print("by source:")
for k,(v,n) in sorted(by_src.items(),key=lambda x:-x[1][1]):
    print(f"  {k:10} tone_i2 {v:.3f}  ({n} TBUs)")
by_spk=pooled_by("speaker")
print(f"\nby speaker ({len(by_spk)} speakers, sorted by tone_i2):")
for k,(v,n) in sorted(by_spk.items(),key=lambda x:(x[1][0] if x[1][0]==x[1][0] else -1)):
    if n>=20: print(f"  spk {k:>6}  tone_i2 {v:.3f}  ({n} TBUs)")
spk_means=[v for v,n in by_spk.values() if v==v and n>=20]
if spk_means:
    import numpy as np
    print(f"\nacross speakers (>=20 TBUs): mean {np.mean(spk_means):.3f}  "
          f"min {np.min(spk_means):.3f}  max {np.max(spk_means):.3f}  (n_spk={len(spk_means)})")

## 9. Plot + save the anchor to S3

In [ ]:
import matplotlib.pyplot as plt, numpy as np, json, datetime
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(clip_scores,bins=20,color="#4C78A8",alpha=0.75,edgecolor="white")
ax.axvspan(lo,hi,color="#F58518",alpha=0.25,label=f"95% CI [{lo:.3f}, {hi:.3f}]")
ax.axvline(point,color="#F58518",lw=2.5,label=f"native (pooled) = {point:.3f}")
ax.axvline(0.58,color="green",ls="--",lw=1.5,label="old hand-set anchor 0.58")
ax.axvline(1/3,color="grey",ls=":",lw=1.5,label="chance 0.333")
ax.set_xlabel("tone_i2 (balanced H/M/L recall)"); ax.set_ylabel("native clips")
ax.set_title(f"Native Yorùbá tone_i2 ceiling  ({len(records)} clips, {len({r['speaker'] for r in records})} speakers)")
ax.legend(fontsize=8); plt.tight_layout()
png=f"{WORK}/native_anchor.png"; plt.savefig(png,dpi=130)
s3.upload_file(png,BUCKET,"tts_data/yoruba/tone_v2/native_anchor.png"); plt.show()

OUT=dict(
    date=datetime.date.today().isoformat(),
    manifest=BIBLE_MANIFEST,
    n_clips=len(records), n_speakers=len({r["speaker"] for r in records}),
    n_tbu_scored=int(sum(r["n_scored"] for r in records)),
    tone_i2_pooled=point, ci95=[lo,hi], half_width=half,
    tone_i2_clip_mean=clip_mean, tone_i2_clip_std=clip_std,
    per_class_recall=per_class, per_class_support=per_class_support,
    mean_coverage=cov_mean, native_cer_mean=cer_mean,
    by_source={k:dict(tone_i2=v,n_tbu=n) for k,(v,n) in by_src.items()},
    calibration=dict(theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,late_frac=I2_LATE),
    n_boot=N_BOOT, ci_alpha=ALPHA, boot_seed=BOOT_SEED,
    method=("clip-level bootstrap of pooled balanced per-class (H/M/L) recall; identical f0_abs_score call as the "
            "model evals (MMS-yor aligner, mid_ref=None, frozen f0_abs_calibration.v1.json). Native audio is "
            "deterministic -> no generation-seed variance; CI reflects clip/speaker sampling only."),
)
s3.put_object(Bucket=BUCKET,Key=ANCHOR_KEY,Body=json.dumps(OUT,ensure_ascii=False,indent=2).encode())
print(f"-> s3://{BUCKET}/{ANCHOR_KEY}")
print(f"-> s3://{BUCKET}/tts_data/yoruba/tone_v2/native_anchor.png")

## 10. Listen — a few native clips (inline)

In [ ]:
import IPython.display as ipd, soundfile as sf
for r in records[:3]:
    w=_read([c for c in clips if c["id"]==r["id"]][0]["path"])
    print(f"{r['id']} | {r['source']} spk {r['speaker']} | tone_i2 {r['tone_i2_clip']:.2f} cov {r['coverage']:.2f} | {r['text'][:60]}")
    ipd.display(ipd.Audio(w,rate=SR))

## 11. Drop it onto the poster

The print in §7 gives a `native = X ± Y` you can paste directly. Concretely, update **`poster.tex`**:

1. **Result-1 footnote** — replace the bare `native\,$\approx$\,0.58` with the measured value + provenance, e.g.
   `native\,$=$\,X.XX\,$\pm$\,Y.YY (N clips, M speakers, 95\% CI)`.
2. **Result-1 bullet** — "near-native tone zero-shot (tone\_i2 0.57 vs native 0.58)" → use the measured anchor.
3. **Result-1 saturation note** and any `native ceiling $\approx$0.58` → the measured number.
4. (Optional) add one line: *"the native ceiling itself is `X.XX ± Y.YY` — `tone_i2` reads real native speech
   well below 1.0, so values near it are saturation, not error."* This turns the reviewer's biggest objection into
   a stated, measured fact.

Tell me the printed `native = X ± Y` (or just say "done") and I'll make the exact `poster.tex` edits + commit.